# Notebook 04: Apprentissage Semi-Supervisé

Ce notebook combine des labels faibles issus du clustering avec un petit sous-ensemble de labels experts pour simuler un entraînement semi-supervisé.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Chargement des données

In [ ]:
weak_labels_path = DATA_DIR / 'processed' / 'weak_labels_kmeans.npy'
if weak_labels_path.exists():
    weak_labels = np.load(weak_labels_path)
else:
    np.random.seed(42)
    weak_labels = np.random.randint(0, 2, 500)

n_samples = len(weak_labels)
n_features = 128
np.random.seed(42)
X = np.random.randn(n_samples, n_features).astype(np.float32)

true_labels = np.random.randint(0, 2, n_samples)
print('X shape:', X.shape)
print('weak_labels shape:', weak_labels.shape)

## 2. Split et labels experts

In [ ]:
indices = np.arange(n_samples)
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

n_expert = max(20, int(0.1 * len(train_idx)))
expert_indices = np.random.choice(train_idx, size=n_expert, replace=False)

train_labels = weak_labels.copy()
train_labels[expert_indices] = true_labels[expert_indices]

print('Train:', len(train_idx))
print('Val:', len(val_idx))
print('Test:', len(test_idx))
print('Labels experts:', len(expert_indices))

## 3. DataLoaders

In [ ]:
def make_loader(x_idx, y):
    x_tensor = torch.tensor(X[x_idx], dtype=torch.float32)
    y_tensor = torch.tensor(y[x_idx], dtype=torch.long)
    ds = TensorDataset(x_tensor, y_tensor)
    return DataLoader(ds, batch_size=32, shuffle=True)

train_loader = make_loader(train_idx, train_labels)
val_loader = make_loader(val_idx, true_labels)
test_loader = make_loader(test_idx, true_labels)

print('✅ DataLoaders prêts')

## 4. Modèle

In [ ]:
class SimpleSemiSupervisedNet(nn.Module):
    def __init__(self, input_dim=128, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        return self.net(x)

model = SimpleSemiSupervisedNet(input_dim=n_features).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)

## 5. Entraînement

In [ ]:
def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(xb)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(y_true)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, acc

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(5):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1}/5 - train_acc={train_acc:.3f} - val_acc={val_acc:.3f}')

## 6. Évaluation finale

In [ ]:
test_loss, test_acc = run_epoch(test_loader, train=False)
print('Test accuracy:', test_acc)

baseline_accuracy = 0.75
improvement = float(test_acc - baseline_accuracy)
print('Improvement vs baseline:', improvement)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## 7. Sauvegarde des résultats

In [ ]:
semi_supervised_results = {
    'semi_accuracy': float(test_acc),
    'baseline_accuracy': float(baseline_accuracy),
    'improvement': float(improvement),
    'expert_ratio': float(len(expert_indices) / len(train_idx)),
    'model_info': {
        'type': 'SimpleSemiSupervisedNet',
        'input_dim': n_features,
        'n_classes': 2
    }
}

results_path = RESULTS_DIR / 'semi_supervised_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(semi_supervised_results, f, indent=2, ensure_ascii=False)

model_path = MODELS_DIR / 'best_semi_supervised_model.pth'
torch.save(model.state_dict(), model_path)

print('Résultats sauvegardés dans:', results_path)
print('Modèle sauvegardé dans:', model_path)